# Lasso Regression (L1 Regularization)

In this notebook we build a Lasso regression model to predict profit based on hours worked. Unlike Ridge (L2), Lasso uses L1 regularization which can shrink some coefficients to exactly zero — effectively performing feature selection.

In [1]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import  StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso,LassoCV
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from joblib import dump, load   

## Define the Dataset

Same dataset as the Ridge notebook: 9 samples with `hours` as the feature and `profit` as the target.

In [2]:
hours = np.array([1.5, 2, 3, 4, 5, 6, 7, 8, 10]).reshape(-1, 1)
profit = np.array([40, 50, 70, 90, 100, 110, 130, 145, 180]).reshape(-1, 1)

## Create Polynomial Features

We generate polynomial features up to degree 3 (hours, hours², hours³). This is one degree less than the Ridge notebook, keeping the model simpler.

In [3]:
poly_con = PolynomialFeatures(degree=3,include_bias=False)
poly_hours = poly_con.fit_transform(hours)

## Train/Test Split & Scaling

We split the data (80/20) and apply `StandardScaler` (zero mean, unit variance) instead of the `Normalizer` used in the Ridge notebook. StandardScaler is fitted on training data only, then applied to both sets to prevent data leakage.

In [4]:
poly_hours_train, poly_hours_test, profit_train, profit_test = train_test_split(
    poly_hours, profit, test_size=0.2, random_state=101
)
scaler = StandardScaler()
scaler.fit(poly_hours_train)
scaled_poly_hours_train = scaler.transform(poly_hours_train)
scaled_poly_hours_test = scaler.transform(poly_hours_test)

## LassoCV: Cross-Validated Alpha Selection

`LassoCV` automatically tests 200 alpha values and selects the one with the lowest cross-validation error. The `eps` parameter controls the ratio between the smallest and largest alpha values.

Results: MAE ≈ 1.4, RMSE ≈ 2.0 — strong performance on the test set.

In [5]:
l1_model = LassoCV(eps=0.1, n_alphas=200)
l1_model.fit(scaled_poly_hours_train, profit_train)
profit_predictions = l1_model.predict(scaled_poly_hours_test)
mae = mean_absolute_error(profit_test, profit_predictions)
rmse = root_mean_squared_error(profit_test, profit_predictions)
print(f"MAE: {mae} ")
print(f"RMSE: {rmse} ")

MAE: 1.445070067427153 
RMSE: 1.956629006069772 


c:\Users\Adir\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(
c:\Users\Adir\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:1756: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


## Inspecting Coefficients — Feature Selection in Action

The coefficients are `[39.39, 0, 0]`, meaning Lasso has zeroed out hours² and hours³. This is the key difference from Ridge: **Lasso performs automatic feature selection** by eliminating features that don't contribute enough, keeping only the linear term.

In [6]:
l1_model.coef_

array([39.3885776,  0.       ,  0.       ])

## Alpha Values Tested by LassoCV

These are the 200 alpha values that `LassoCV` evaluated, ranging from ~43.9 down to ~4.4. The model internally selected the alpha that minimized the cross-validation error.

In [7]:
l1_model.alphas_

array([43.9375256 , 43.43206412, 42.93241751, 42.43851889, 41.95030212,
       41.46770185, 40.99065345, 40.51909306, 40.05295755, 39.59218451,
       39.13671225, 38.68647978, 38.24142683, 37.80149381, 37.36662182,
       36.93675264, 36.51182871, 36.09179316, 35.67658973, 35.26616284,
       34.86045754, 34.45941951, 34.06299506, 33.67113112, 33.28377522,
       32.90087549, 32.52238069, 32.14824012, 31.7784037 , 31.41282191,
       31.05144582, 30.69422702, 30.34111771, 29.9920706 , 29.64703896,
       29.3059766 , 28.96883785, 28.63557757, 28.30615116, 27.9805145 ,
       27.65862399, 27.34043654, 27.02590956, 26.71500092, 26.407669  ,
       26.10387266, 25.80357122, 25.50672447, 25.21329268, 24.92323656,
       24.63651727, 24.35309643, 24.07293608, 23.79599873, 23.52224729,
       23.25164512, 22.98415598, 22.71974406, 22.45837395, 22.20001068,
       21.94461964, 21.69216664, 21.44261789, 21.19593997, 20.95209986,
       20.7110649 , 20.47280284, 20.23728176, 20.00447014, 19.77

## Manual Alpha Sweep

Here we manually loop over each alpha value from the `LassoCV` grid, train a separate Lasso model for each one, and print MAE and RMSE on the test set. This lets us see how the error changes as alpha decreases (less regularization).

In [8]:
for running_alpha in l1_model.alphas_:
    lasso = Lasso(alpha=running_alpha)
    lasso.fit(scaled_poly_hours_train, profit_train)
    predictions = lasso.predict(scaled_poly_hours_test)
    mae = mean_absolute_error(profit_test,predictions)
    rmse = root_mean_squared_error(profit_test,predictions)
    print(f"Alpha={running_alpha}\nMAE={mae}\nRMSE={rmse}\n\n")

Alpha=43.937525601139086
MAE=37.5
RMSE=38.242646351945886


Alpha=43.43206411773231
MAE=37.5
RMSE=38.242646351945886


Alpha=42.93241751142003
MAE=36.57995911616786
RMSE=37.309549253932865


Alpha=42.43851888730182
MAE=36.12786154446866
RMSE=36.85104277972885


Alpha=41.950302120042466
MAE=35.6809649463361
RMSE=36.39781561574704


Alpha=41.46770184501903
MAE=35.23920948927446
RMSE=35.949807093190884


Alpha=40.990653449569336
MAE=34.802536029106676
RMSE=35.50695724225777


Alpha=40.51909306434145
MAE=34.37088610205598
RMSE=35.06920678411855


Alpha=40.052957554742605
MAE=33.94420191691851
RMSE=34.63649712298973


Alpha=39.592184512486334
MAE=33.5224263473258
RMSE=34.20877033829704


Alpha=39.13671224723712
MAE=33.10550292409671
RMSE=33.785969176930315


Alpha=38.68647977835089
MAE=32.69337582767687
RMSE=33.36803704558734


Alpha=38.24142682671073
MAE=32.28598988066537
RMSE=32.954918003206785


Alpha=37.80149380665646
MAE=31.883290540427424
RMSE=32.5465567534886


Alpha=37.3666218180069